# Sterowanie trajektorią w modelach generatywnych

![Kot i dyfuzja](img/img.png)

W modelach generatywnych (dyfuzyjnych i opartych na algorytmach typu *flow-matching*) obraz nie powstaje w jednym kroku. Proces ten określa **trajektoria** – sekwencja stanów prowadząca od początkowego *noise'u* $x_T$ do gotowego obrazu $x_0$. Każdy *sampling step* aktualizuje dane, bezpośrednio decydując o ostatecznym wyglądzie grafiki.

## Przebieg procesu (Pipeline)

Trajektoria fizycznie realizuje się poprzez wieloetapowy *denoising*. Standardowy schemat tego procesu (*pipeline*) wygląda następująco:

1. **Inicjalizacja:** Proces startuje od losowego *Gaussian noise* $x_T \sim \mathcal{N}(0, \mathbf{I})$. W zadaniach edycyjnych (*image-to-image*) wejściem jest oryginalny obraz zaszumiony do ustalonego poziomu $t < T$.
2. **Conditioning:** Model przyjmuje instrukcję $c$ (np. *embedding* promptu z *text encodera*), która wyznacza kierunek *denoisingu* w rozkładzie $p_\theta(x_{t-1} | x_t, c)$.
3. **Denoising loop:** Sieć neuronowa $\epsilon_\theta$ estymuje ilość *noise'u* w danych $x_t$, a dedykowany *scheduler* go redukuje. Powstaje ciąg stanów $\{x_T, x_{T-1}, \dots, x_0\}$, tworzący fizyczny kształt trajektorii.
4. **Decoding:** Po zakończeniu pętli, *decoder* (np. model VAE) zamienia skompresowane dane z *latent space* na finalne piksele obrazu.

## Wpływ trajektorii na obraz

Trajektoria wyznacza stany pośrednie, co wprost decyduje o procesie powstawania obrazu:

* **Ogólny zarys:** W początkowych krokach (dla dużych $t$) trajektoria ustala kompozycję, proporcje i układ głównych elementów sceny.
* **Tworzenie detali:** W krokach końcowych ($t \to 0$) algorytm generuje drobne szczegóły. Inna trajektoria przy tym samym wejściowym *noise* $x_T$ i warunku $c$ wygeneruje inne tekstury (np. inny układ porów na skórze czy refleksów światła).
* **Conditioning strength:** Trajektoria określa, jak silnie warunek $c$ wpływa na wynik. W fazie *inference time* kontroluje to parametr *CFG Scale* (Classifier-Free Guidance). Ostateczny wektor $\tilde{\epsilon}_\theta$ oblicza się ze wzoru:
  $$\tilde{\epsilon}_\theta(x_t, c) = (1 - w)\epsilon_\theta(x_t) + w\epsilon_\theta(x_t, c)$$
  Waga $w$ określa, jak mocno model podąża za promptem zamiast generować własne, losowe elementy.

## Kształt trajektorii

Z punktu widzenia inżynierii obliczeniowej, trajektoria jest zawsze realizowana w sekwencji dyskretnych kroków. O wydajności i precyzji modelu decyduje jednak fizyczny **kształt ścieżki** łączącej początkowy *noise* z docelowym obrazem.

Rozróżniamy dwa główne podejścia do kształtowania trajektorii:

* **Curved trajectories (nieliniowe):** W klasycznych modelach dyfuzyjnych naturalna ścieżka *denoisingu* meandruje przez złożone rozkłady danych. Algorytm musi nieustannie korygować kierunek, co wymusza podział trasy na wiele krótkich kroków (zazwyczaj 30-100). Skutkuje to wysokim kosztem obliczeniowym, długim *inference time* oraz ryzykiem powstawania wizualnych artefaktów na ostrych załamaniach trajektorii.
* **Rectified trajectories (liniowe):** Nowoczesne architektury (*Rectified Flow*, *Flow Matching*) wprowadzają matematyczne prostowanie wektorów. Algorytm wytycza najkrótszą, liniową ścieżkę w *state space* między szumem $x_T$ a obrazem $x_0$. Zmniejsza to ryzyko błędów i pozwala zredukować proces do zaledwie kilku długich kroków (często 1-4). Efektem jest błyskawiczny *inference time* i ścisła spójność z promptem, gdyż model zmierza bezpośrednio do celu.

## Co można osiągnąć za pomocą sterowania trajektorią?

Świadome manipulowanie przebiegiem trajektorii to fundament zaawansowanej pracy z generatywną sztuczną inteligencją. Pozwala ono na osiągnięcie konkretnych rezultatów inżynieryjnych i wizualnych:

* **Few-step generation:** Zastosowanie modeli o wyprostowanych trajektoriach pozwala zredukować liczbę *sampling steps* z kilkudziesięciu do kilku, generując obrazy w czasie rzeczywistym przy minimalnym zużyciu zasobów obliczeniowych (VRAM).
* **Inpainting / Image-to-image:** Manipulując długością trajektorii, decydujemy o stopniu modyfikacji obrazu. Krótka trajektoria zachowuje oryginalną kompozycję i tożsamość obiektu (zmieniając np. tylko oświetlenie), podczas gdy długa trajektoria pozwala na całkowitą przebudowę sceny.
* **Separacja zarysu od detali:** Trajektoria wyznacza stany pośrednie. W początkowych krokach ustala układ głównych elementów sceny, a w końcowych ($t \to 0$) generuje drobne szczegóły. Odpowiednie zarządzanie tymi etapami zapobiega zjawisku wizualnego przeładowania (tzw. *overbaking*).
* **Redukcja artefaktów i halucynacji:** Gładka, dobrze ukształtowana ścieżka zapobiega błędom strukturalnym. Model rzadziej generuje błędną anatomię (np. niepoprawną liczbę palców) lub niefizyczne połączenia obiektów.

## Narzędzia kontroli: Jak wpływamy na trajektorię?

Kontrola ścieżki *denoisingu* zachodzi na dwóch głównych etapach życia modelu i wykorzystuje ściśle określone mechanizmy, od modyfikacji architektury po nawigację w czasie rzeczywistym:

### 1. Training time
W fazie *training time* aktualizowane są *weights* modelu $\theta$, aby algorytm minimalizował *noise prediction loss*.
* **Loss function & Regularization:** *Dataset* i *loss function* projektuje się tak, aby wymusić proste ścieżki. Dodatkowe kary matematyczne zapobiegają niefizycznym załamaniom trajektorii.
* **Distillation:** Uczenie mniejszego modelu (tzw. *student model*) naśladowania stabilnej trajektorii większego modelu (*teacher model*) w celu redukcji niezbędnych kroków.
* **Control Adapters (np. ControlNet) & LoRA:** Dodatkowe sieci neuronowe lub celowane modyfikacje *weights*, które twardo zawężają trajektorię do zadanej geometrii (np. map głębi) lub konkretnego stylu malarskiego, nie niszcząc bazowej wiedzy modelu.

### 2. Inference time
W fazie generacji użytkownik modyfikuje przebieg fizycznej trajektorii w *latent space* korzystając z zamrożonego modelu:
* **Scheduler & Sampling Steps ($N$):** Wybór algorytmu rozwiązującego równania (np. Euler, DDIM) dopasowanego do krzywizny trajektorii oraz ustalenie gęstości jej podziału na kroki.
* **CFG Scale ($w$) & Negative Prompting:** Skalowanie siły, z jaką trajektoria podąża za warunkiem $c$. Dodatkowo, *negative prompt* matematycznie odpycha ścieżkę od niepożądanych rejonów (np. błędnej anatomii).
* **Denoising Strength:** Używany przy modyfikacji istniejących obrazów. Określa punkt startowy trajektorii ($t < T$) na osi czasu – decyduje, czy model zachowa oryginalną strukturę, czy zniszczy ją na rzecz wygenerowania nowej.
* **Prompt Scheduling & Regional Prompting:** Dynamiczna zmiana warunku tekstowego w trakcie pętli (np. budowa zarysu w pierwszych 50% kroków z jednym promptem, a detali w pozostałych z innym) lub wytyczanie osobnych trajektorii dla różnych stref obrazu i ich spajanie (tzw. *blending*).
